**Mi primer modelo de Machine Learnig:**
Estimacion de tiempos de preparacion de productos
FASE 1. prepacion de datos

In [1]:

# ==========================================
# FASE 1: PREPARACIÓN DE DATOS
# ==========================================

# Importar la biblioteca Pandas
import pandas as pd

# Importar NumPy
import numpy as np

# Herramienta para cargar archivos desde el computador
from google.colab import files

print("Bibliotecas importadas correctamente.")


Bibliotecas importadas correctamente.


In [3]:

# Cargar el archivo CSV desde el computador
archivos = files.upload()


Saving base_datos_machine_learning_mesero_virtual.csv to base_datos_machine_learning_mesero_virtual (1).csv


In [4]:

# Leer el archivo CSV
df = pd.read_csv(
    "base_datos_machine_learning_mesero_virtual.csv"
)

# Mostrar los primeros cinco registros
print("Primeros registros de nuestra base de datos:")

display(df.head())


Primeros registros de nuestra base de datos:


,Producto,Categoria,Dia_semana,Hora_pedido,Cantidad,Pedidos_pendientes,Nivel_ocupacion,Personal_cocina,Tiempo_preparacion_min
0,Hamburguesa clásica,Comidas rápidas,Domingo,13:15,3,10,Alta,2,34
1,Hamburguesa especial,Comidas rápidas,Sábado,17:15,2,5,Media,3,27
2,Hamburguesa especial,Comidas rápidas,Jueves,13:00,3,7,Media,2,37
3,Jugo natural,Bebidas,Martes,21:15,1,3,Baja,2,7
4,Salchipapa,Comidas rápidas,Martes,13:00,2,10,Alta,2,29


In [5]:

# Consultar dimensiones de la base de datos
print("Cantidad de registros y columnas:")
print(df.shape)

# Consultar información de las columnas
print("\nInformación de la base de datos:")
df.info()

# Identificar valores vacíos
print("\nValores vacíos por columna:")
print(df.isnull().sum())


Cantidad de registros y columnas:
(400, 9)

Información de la base de datos:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Producto                400 non-null    object
 1   Categoria               400 non-null    object
 2   Dia_semana              400 non-null    object
 3   Hora_pedido             400 non-null    object
 4   Cantidad                400 non-null    int64 
 5   Pedidos_pendientes      400 non-null    int64 
 6   Nivel_ocupacion         400 non-null    object
 7   Personal_cocina         400 non-null    int64 
 8   Tiempo_preparacion_min  400 non-null    int64 
dtypes: int64(4), object(5)
memory usage: 28.3+ KB

Valores vacíos por columna:
Producto                  0
Categoria                 0
Dia_semana                0
Hora_pedido               0
Cantidad                  0
Pedidos_pendientes        0

In [6]:

# Consultar los nombres de las columnas
print("Variables disponibles en nuestra base de datos:")

for columna in df.columns:
    print("-", columna)

# Consultar los tipos de datos
print("\nTipos de datos:")
print(df.dtypes)


Variables disponibles en nuestra base de datos:
- Producto
- Categoria
- Dia_semana
- Hora_pedido
- Cantidad
- Pedidos_pendientes
- Nivel_ocupacion
- Personal_cocina
- Tiempo_preparacion_min

Tipos de datos:
Producto                  object
Categoria                 object
Dia_semana                object
Hora_pedido               object
Cantidad                   int64
Pedidos_pendientes         int64
Nivel_ocupacion           object
Personal_cocina            int64
Tiempo_preparacion_min     int64
dtype: object


In [7]:

# Separar las variables de entrada
X = df.drop("Tiempo_preparacion_min", axis=1)

# Definir la variable que queremos predecir
y = df["Tiempo_preparacion_min"]

print("Variables de entrada:")
display(X.head())

print("\nVariable que queremos predecir:")
display(y.head())


Variables de entrada:


,Producto,Categoria,Dia_semana,Hora_pedido,Cantidad,Pedidos_pendientes,Nivel_ocupacion,Personal_cocina
0,Hamburguesa clásica,Comidas rápidas,Domingo,13:15,3,10,Alta,2
1,Hamburguesa especial,Comidas rápidas,Sábado,17:15,2,5,Media,3
2,Hamburguesa especial,Comidas rápidas,Jueves,13:00,3,7,Media,2
3,Jugo natural,Bebidas,Martes,21:15,1,3,Baja,2
4,Salchipapa,Comidas rápidas,Martes,13:00,2,10,Alta,2



Variable que queremos predecir:


,Tiempo_preparacion_min
0,34
1,27
2,37
3,7
4,29


In [8]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Identificar las columnas de texto
columnas_categoricas = [
    "Producto",
    "Categoria",
    "Dia_semana",
    "Nivel_ocupacion"
]

# Identificar las columnas numéricas
columnas_numericas = [
    "Cantidad",
    "Pedidos_pendientes",
    "Personal_cocina"
]

# Convertir la hora a número decimal
X = X.copy()

X["Hora_pedido"] = (
    pd.to_datetime(X["Hora_pedido"], format="%H:%M")
    .dt.hour
    + pd.to_datetime(X["Hora_pedido"], format="%H:%M")
    .dt.minute / 60
)

columnas_numericas.append("Hora_pedido")

# Preparar el transformador
preprocesador = ColumnTransformer(
    transformers=[
        ("categoricas", OneHotEncoder(handle_unknown="ignore"),
         columnas_categoricas),
        ("numericas", "passthrough", columnas_numericas)
    ]
)

print("Variables categóricas y numéricas preparadas.")


Variables categóricas y numéricas preparadas.


In [9]:

# Importar herramienta para dividir los datos
from sklearn.model_selection import train_test_split

# Dividir los datos
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Mostrar las dimensiones
print("Registros para entrenamiento:", X_entrenamiento.shape[0])
print("Registros para prueba:", X_prueba.shape[0])


Registros para entrenamiento: 320
Registros para prueba: 80


In [10]:

# Importar el algoritmo Random Forest
from sklearn.ensemble import RandomForestRegressor

# Crear el modelo completo
from sklearn.pipeline import Pipeline

modelo = Pipeline(steps=[
    ("preprocesamiento", preprocesador),
    ("regresion", RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

# Entrenar el modelo
modelo.fit(X_entrenamiento, y_entrenamiento)

print("¡Modelo de Machine Learning entrenado correctamente!")


¡Modelo de Machine Learning entrenado correctamente!


In [11]:

# Generar predicciones con los datos de prueba
predicciones = modelo.predict(X_prueba)

# Comparar los tiempos reales con los estimados
comparacion = pd.DataFrame({
    "Tiempo_real": y_prueba.values,
    "Tiempo_estimado": np.round(predicciones, 2)
})

display(comparacion.head(10))


,Tiempo_real,Tiempo_estimado
0,8,13.58
1,19,21.23
2,26,27.00
3,12,14.42
4,20,16.89
5,19,15.38
6,34,30.96
7,39,37.14
8,21,20.59
9,46,42.64


In [12]:

from sklearn.metrics import mean_absolute_error, r2_score

# Calcular las métricas
mae = mean_absolute_error(y_prueba, predicciones)
r2 = r2_score(y_prueba, predicciones)

print("RESULTADOS DE LA EVALUACIÓN")
print("---------------------------")
print(f"Error promedio (MAE): {mae:.2f} minutos")
print(f"Coeficiente de determinación (R²): {r2:.2f}")


RESULTADOS DE LA EVALUACIÓN
---------------------------
Error promedio (MAE): 2.44 minutos
Coeficiente de determinación (R²): 0.91


In [13]:

# ==========================================
# FASE 5: PREDICCIÓN DE UN PRODUCTO NUEVO
# ==========================================

# Crear los datos del nuevo pedido
nuevo_pedido = pd.DataFrame([{
    "Producto": "Hamburguesa clásica",
    "Categoria": "Comidas rápidas",
    "Dia_semana": "Sábado",
    "Hora_pedido": "20:00",
    "Cantidad": 1,
    "Pedidos_pendientes": 10,
    "Nivel_ocupacion": "Alta",
    "Personal_cocina": 2
}])

# Convertir la hora a formato numérico
nuevo_pedido["Hora_pedido"] = (
    pd.to_datetime(nuevo_pedido["Hora_pedido"], format="%H:%M").dt.hour
    + pd.to_datetime(nuevo_pedido["Hora_pedido"], format="%H:%M").dt.minute / 60
)

# Obtener la predicción
tiempo_estimado = modelo.predict(nuevo_pedido)

# Mostrar el resultado
print("PREDICCIÓN DEL TIEMPO DE PREPARACIÓN")
print("------------------------------------")
print("Producto: Hamburguesa clásica")
print("Día: Sábado")
print("Hora: 20:00")
print("Pedidos pendientes: 10")
print("Nivel de ocupación: Alta")
print(f"Tiempo estimado: {tiempo_estimado[0]:.2f} minutos")


PREDICCIÓN DEL TIEMPO DE PREPARACIÓN
------------------------------------
Producto: Hamburguesa clásica
Día: Sábado
Hora: 20:00
Pedidos pendientes: 10
Nivel de ocupación: Alta
Tiempo estimado: 21.21 minutos


In [14]:
# ==========================================
# FASE 6: PRUEBA CON MENOR DEMANDA
# ==========================================

nuevo_pedido_2 = pd.DataFrame([{
    "Producto": "Hamburguesa clásica",
    "Categoria": "Comidas rápidas",
    "Dia_semana": "Lunes",
    "Hora_pedido": "10:00",
    "Cantidad": 1,
    "Pedidos_pendientes": 1,
    "Nivel_ocupacion": "Baja",
    "Personal_cocina": 3
}])

nuevo_pedido_2["Hora_pedido"] = (
    pd.to_datetime(
        nuevo_pedido_2["Hora_pedido"],
        format="%H:%M"
    ).dt.hour
)

tiempo_estimado_2 = modelo.predict(nuevo_pedido_2)

print("SEGUNDA PRUEBA DE PREDICCIÓN")
print("------------------------------------")
print("Producto: Hamburguesa clásica")
print("Día: Lunes")
print("Hora: 10:00")
print("Pedidos pendientes: 1")
print("Nivel de ocupación: Baja")
print("Personal de cocina: 3")
print(f"Tiempo estimado: {tiempo_estimado_2[0]:.2f} minutos")

SEGUNDA PRUEBA DE PREDICCIÓN
------------------------------------
Producto: Hamburguesa clásica
Día: Lunes
Hora: 10:00
Pedidos pendientes: 1
Nivel de ocupación: Baja
Personal de cocina: 3
Tiempo estimado: 16.52 minutos


In [15]:
# ==========================================
# FASE 7: OPCIONES DISPONIBLES
# ==========================================

print("PRODUCTOS DISPONIBLES:")
print(df["Producto"].unique())

print("\nCATEGORÍAS DISPONIBLES:")
print(df["Categoria"].unique())

print("\nDÍAS DISPONIBLES:")
print(df["Dia_semana"].unique())

print("\nNIVELES DE OCUPACIÓN:")
print(df["Nivel_ocupacion"].unique())

PRODUCTOS DISPONIBLES:
['Hamburguesa clásica' 'Hamburguesa especial' 'Jugo natural' 'Salchipapa'
 'Bandeja típica' 'Ensalada' 'Perro caliente' 'Arepa con queso'
 'Pizza mediana' 'Pizza personal']

CATEGORÍAS DISPONIBLES:
['Comidas rápidas' 'Bebidas' 'Comida típica' 'Comida saludable'
 'Comida italiana']

DÍAS DISPONIBLES:
['Domingo' 'Sábado' 'Jueves' 'Martes' 'Viernes' 'Miércoles' 'Lunes']

NIVELES DE OCUPACIÓN:
['Alta' 'Media' 'Baja']


In [17]:
# ==========================================
# FASE 7: FORMULARIO INTERACTIVO
# ==========================================

print("======================================")
print("   MESERO VIRTUAL - PREDICCIÓN ML")
print("======================================")

producto = input("Nombre del producto: ")
categoria = input("Categoría: ")
dia = input("Día de la semana: ")
hora = input("Hora del pedido (HH:MM): ")

cantidad = int(input("Cantidad de productos: "))
pendientes = int(input("Pedidos pendientes: "))
ocupacion = input("Nivel de ocupación (Baja, Media, Alta): ")
personal = int(input("Personas disponibles en cocina: "))

# Crear registro con los datos ingresados
pedido = pd.DataFrame([{
    "Producto": producto,
    "Categoria": categoria,
    "Dia_semana": dia,
    "Hora_pedido": hora,
    "Cantidad": cantidad,
    "Pedidos_pendientes": pendientes,
    "Nivel_ocupacion": ocupacion,
    "Personal_cocina": personal
}])

# Convertir hora a formato numérico
pedido["Hora_pedido"] = (
    pd.to_datetime(
        pedido["Hora_pedido"],
        format="%H:%M"
    ).dt.hour
    + pd.to_datetime(
        pedido["Hora_pedido"],
        format="%H:%M"
    ).dt.minute / 60
)

# Generar predicción
resultado = modelo.predict(pedido)

print("\n======================================")
print("       RESULTADO DE PREDICCIÓN")
print("======================================")
print(f"Producto: {producto}")
print(f"Tiempo estimado: {resultado[0]:.2f} minutos")
print("======================================")

   MESERO VIRTUAL - PREDICCIÓN ML
Nombre del producto: Hamburguesa clásica
Categoría: Comidas rápidas
Día de la semana: Lunes
Hora del pedido (HH:MM): 10:00
Cantidad de productos: 1
Pedidos pendientes: 1
Nivel de ocupación (Baja, Media, Alta): Baja
Personas disponibles en cocina: 3

       RESULTADO DE PREDICCIÓN
Producto: Hamburguesa clásica
Tiempo estimado: 16.52 minutos


In [18]:

# ==========================================
# FASE 8: FORMULARIO CON NORMALIZACIÓN
# ==========================================

import unicodedata
import pandas as pd

def normalizar(texto):
    texto = str(texto).strip().casefold()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(
        caracter for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )
    return " ".join(texto.split())

def buscar_valor(valor, opciones, nombre):
    for opcion in opciones:
        if normalizar(valor) == normalizar(opcion):
            return opcion

    raise ValueError(
        f"{nombre} no reconocido. Opciones válidas: {list(opciones)}"
    )

print("======================================")
print("   MESERO VIRTUAL - PREDICCIÓN ML")
print("======================================")

# Obtener opciones reales del entrenamiento
productos = df["Producto"].unique()
categorias = df["Categoria"].unique()
dias = df["Dia_semana"].unique()
ocupaciones = df["Nivel_ocupacion"].unique()

try:
    producto = buscar_valor(
        input("Nombre del producto: "), productos, "Producto"
    )

    categoria = buscar_valor(
        input("Categoría: "), categorias, "Categoría"
    )

    dia = buscar_valor(
        input("Día de la semana: "), dias, "Día"
    )

    hora = input("Hora del pedido (HH:MM): ").strip()
    hora_convertida = pd.to_datetime(
        hora, format="%H:%M", errors="raise"
    )

    cantidad = int(input("Cantidad de productos: "))
    pendientes = int(input("Pedidos pendientes: "))

    ocupacion = buscar_valor(
        input("Nivel de ocupación (Baja, Media, Alta): "),
        ocupaciones,
        "Ocupación"
    )

    personal = int(
        input("Personas disponibles en cocina: ")
    )

    if cantidad < 1 or pendientes < 0 or personal < 1:
        raise ValueError(
            "Cantidad y personal deben ser positivos; "
            "los pedidos pendientes no pueden ser negativos."
        )

    pedido = pd.DataFrame([{
        "Producto": producto,
        "Categoria": categoria,
        "Dia_semana": dia,
        "Hora_pedido": (
            hora_convertida.hour
            + hora_convertida.minute / 60
        ),
        "Cantidad": cantidad,
        "Pedidos_pendientes": pendientes,
        "Nivel_ocupacion": ocupacion,
        "Personal_cocina": personal
    }])

    resultado = modelo.predict(pedido)

    print("\n======================================")
    print("       RESULTADO DE PREDICCIÓN")
    print("======================================")
    print(f"Producto: {producto}")
    print(f"Tiempo estimado: {resultado[0]:.2f} minutos")
    print("======================================")

except ValueError as error:
    print("\nNo se pudo generar la predicción.")
    print("Detalle:", error)


   MESERO VIRTUAL - PREDICCIÓN ML


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_722/3707163629.py", line 38, in <cell line: 0>
    input("Nombre del producto: "), productos, "Producto"
    ~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/ipykernel/kernelbase.py", line 1177, in raw_input
    return self._input_request(
           ~~~~~~~~~~~~~~~~~~~^
        str(prompt),
        ^^^^^^^^^^^^
    ...<2 lines>...
        password=False,
        ^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/ipykernel/kernelbase.py", line 1219, in _input_request
    raise KeyboardInterrupt("Interrupted by user") from None
KeyboardInterrupt: Interrupted by user

During handling of the above exception, another exception occurred:

Traceback (most recent call las

TypeError: object of type 'NoneType' has no len()

In [19]:
# ==========================================
# FASE 9: GUARDAR EL MODELO ENTRENADO
# ==========================================

import joblib

# Guardar el modelo completo, incluido el preprocesamiento
joblib.dump(modelo, "modelo_mesero_virtual.joblib")

print("Modelo guardado correctamente.")

Modelo guardado correctamente.


In [20]:
from google.colab import files

files.download("modelo_mesero_virtual.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>